# 01 - Data Exploration

This notebook explores the Chest X-Ray Pneumonia dataset:
- Class distribution analysis
- Sample visualization
- Image properties (resolution, aspect ratios)
- Data split statistics

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

from src.data import _collect_image_paths_and_labels, CLASS_NAMES

plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

DATA_ROOT = Path("../data/chest_xray")

## 1. Dataset overview and class distribution

In [ ]:
splits = {}
for split in ["train", "val", "test"]:
    paths, labels = _collect_image_paths_and_labels(DATA_ROOT / split)
    splits[split] = {"paths": paths, "labels": labels}
    counts = Counter(labels)
    total = len(labels)
    print(f"{split:>5s}: {total:5d} images | "
          f"Normal: {counts[0]:4d} ({counts[0]/total*100:.1f}%) | "
          f"Pneumonia: {counts[1]:4d} ({counts[1]/total*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for i, (split, data) in enumerate(splits.items()):
    counts = Counter(data["labels"])
    bars = axes[i].bar(
        CLASS_NAMES,
        [counts[0], counts[1]],
        color=["#4C72B0", "#DD8452"],
    )
    axes[i].set_title(f"{split.capitalize()} Set")
    axes[i].set_ylabel("Count")
    for bar, count in zip(bars, [counts[0], counts[1]]):
        axes[i].text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            str(count), ha="center", fontsize=10,
        )

plt.suptitle("Class Distribution per Split", fontsize=14)
plt.tight_layout()
plt.savefig("../results/class_distribution.png", bbox_inches="tight")
plt.show()

The training set exhibits significant class imbalance with approximately 74% pneumonia cases.
The original validation set has only 16 images, which is insufficient for reliable model selection.
We will re-split the data in the training pipeline (see `src/data.py`).

## 2. Sample images

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(15, 5))

train_paths = splits["train"]["paths"]
train_labels = splits["train"]["labels"]

for row, class_idx in enumerate([0, 1]):
    class_paths = [p for p, l in zip(train_paths, train_labels) if l == class_idx]
    rng = np.random.RandomState(42)
    sampled = rng.choice(class_paths, size=6, replace=False)

    for col, img_path in enumerate(sampled):
        img = Image.open(img_path)
        axes[row, col].imshow(img, cmap="gray")
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(CLASS_NAMES[class_idx], fontsize=12)

plt.suptitle("Sample Chest X-Ray Images", fontsize=14)
plt.tight_layout()
plt.savefig("../results/sample_images.png", bbox_inches="tight")
plt.show()

## 3. Image resolution analysis

In [ ]:
widths, heights = [], []

for path in train_paths:
    img = Image.open(path)
    w, h = img.size
    widths.append(w)
    heights.append(h)

widths = np.array(widths)
heights = np.array(heights)

print(f"Width  - min: {widths.min()}, max: {widths.max()}, "
      f"mean: {widths.mean():.0f}, median: {np.median(widths):.0f}")
print(f"Height - min: {heights.min()}, max: {heights.max()}, "
      f"mean: {heights.mean():.0f}, median: {np.median(heights):.0f}")
print(f"Aspect ratio - mean: {(widths/heights).mean():.3f}, "
      f"std: {(widths/heights).std():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(widths, bins=50, alpha=0.7, label="Width", color="#4C72B0")
axes[0].hist(heights, bins=50, alpha=0.7, label="Height", color="#DD8452")
axes[0].set_xlabel("Pixels")
axes[0].set_ylabel("Count")
axes[0].set_title("Image Dimensions")
axes[0].legend()

axes[1].scatter(widths, heights, alpha=0.1, s=5)
axes[1].set_xlabel("Width")
axes[1].set_ylabel("Height")
axes[1].set_title("Width vs Height")

plt.tight_layout()
plt.savefig("../results/image_dimensions.png", bbox_inches="tight")
plt.show()

## 4. Pixel intensity distribution

In [ ]:
rng = np.random.RandomState(42)
sample_indices = rng.choice(len(train_paths), size=200, replace=False)

normal_pixels, pneumonia_pixels = [], []

for idx in sample_indices:
    img = np.array(Image.open(train_paths[idx]).convert("L").resize((128, 128))).flatten()
    if train_labels[idx] == 0:
        normal_pixels.extend(img)
    else:
        pneumonia_pixels.extend(img)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(normal_pixels, bins=100, alpha=0.5, density=True, label="Normal", color="#4C72B0")
ax.hist(pneumonia_pixels, bins=100, alpha=0.5, density=True, label="Pneumonia", color="#DD8452")
ax.set_xlabel("Pixel Intensity")
ax.set_ylabel("Density")
ax.set_title("Pixel Intensity Distribution by Class")
ax.legend()
plt.tight_layout()
plt.savefig("../results/pixel_distribution.png", bbox_inches="tight")
plt.show()

## 5. Verify the re-split used in training

In [ ]:
from src.data import get_dataloaders

loaders = get_dataloaders(
    str(DATA_ROOT),
    augmentation="none",
    batch_size=32,
    num_workers=0,
)

info = loaders["info"]
print("Re-split dataset statistics:")
print(f"  Train: {info['train_size']} images, class dist: {info['train_class_dist']}")
print(f"  Val:   {info['val_size']} images, class dist: {info['val_class_dist']}")
print(f"  Test:  {info['test_size']} images, class dist: {info['test_class_dist']}")